# Run Dummy Data from Landing to RAW inc. DQ
This notebook runs an Autoloader-based streaming ingestion job to load data from the landing zone into a raw Delta table, applying schema enforcement and DQ checks. It uses widgets for parameters like config path and run mode (e.g., "availableNow" for batch processing).

In [0]:
dbutils.widgets.text("config_path", "")
dbutils.widgets.text("triggered_by", "manual")
dbutils.widgets.text("run_mode", "availableNow")  # availableNow or processingTime
dbutils.widgets.text("processing_time", "5 minutes")  # only used if run_mode=processingTime

config_path = dbutils.widgets.get("config_path")
triggered_by = dbutils.widgets.get("triggered_by")
run_mode = dbutils.widgets.get("run_mode")
processing_time = dbutils.widgets.get("processing_time")

if not config_path:
    raise ValueError("config_path widget is empty. Provide the YAML config path created by 00_generate_dummy_landing_data.")

In [0]:
dbutils.widgets.dropdown(
    "test_folder",
    "good",
    ["good", "few_rows", "required_nulls", "high_null_close", "all"],
    "Select test subfolder"
)

In [0]:
display(config_path)

print(dbutils.fs.head(config_path, 20000))

'abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/config/lse_prices_test.yml'


pipeline_name: "macro-platform"
source_name: "LSE"
dataset_name: "equities_prices_test"
format: "csv"

landing_path: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/"
raw_table: "raw.lse_equities_prices_test"

# Put schema + checkpoint under UC root too (so you don't hit permission errors)
raw_checkpoint: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/"
schema_location: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/"

read_options:
  header: "true"
  delimiter: ","
  inferSchema: "false"

schema:
  - {name: isin, type: string, nullable: false}
  - {name: trade_date, type: date, nullable: false}
  - {name: close_price, type: decimal(18,6), nullable: true}
  - {name: currency, type: string, nullable: t

In [0]:
import yaml

cfg = yaml.safe_load(dbutils.fs.head(config_path, 20000))
print(cfg["schema"])

[{'name': 'isin', 'type': 'string', 'nullable': False}, {'name': 'trade_date', 'type': 'date', 'nullable': False}, {'name': 'close_price', 'type': 'decimal(18', '6)': None, 'nullable': True}, {'name': 'currency', 'type': 'string', 'nullable': True}]


In [0]:
import yaml

cfg_text = dbutils.fs.head(config_path, 100000)
cfg = yaml.safe_load(cfg_text)

pipeline_name = cfg["pipeline_name"]
source_name   = cfg["source_name"]
dataset_name  = cfg["dataset_name"]
fmt           = cfg["format"]

landing_path  = cfg["landing_path"]
raw_table     = cfg["raw_table"]
checkpoint    = cfg["raw_checkpoint"]
schema_loc    = cfg["schema_location"]

read_opts     = cfg.get("read_options", {})

dq_cfg = cfg.get("dq", {})
stop_on_sev = dq_cfg.get("stop_on_severity", "ERROR")
expected_min_rows = int(dq_cfg.get("expected_min_rows", 0))
required_cols = dq_cfg.get("required_columns", [])
null_thresholds = dq_cfg.get("null_thresholds", {})
policies = dq_cfg.get("policies", {})

print("Loaded config:")
print(" landing_path :", landing_path)
print(" raw_table    :", raw_table)
print(" checkpoint   :", checkpoint)
print(" schema_loc   :", schema_loc)

Loaded config:
 landing_path : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/
 raw_table    : raw.lse_equities_prices_test
 checkpoint   : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/
 schema_loc   : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/


In [0]:
selected = dbutils.widgets.get("test_folder")
base_checkpoint = checkpoint
base_schema_loc = schema_loc


if selected != "all":
    landing_path = f"{landing_path.rstrip('/')}/{selected}/"
    print(f"Overriding landing_path to: {landing_path}")

    # 2) state isolation (this is the key fix)
    checkpoint = f"{base_checkpoint.rstrip('/')}/{selected}/"
    print(f"Overriding checkpoint to: {checkpoint}")
    schema_loc = f"{base_schema_loc.rstrip('/')}/{selected}/"
    print(f"Overriding schema_loc to: {schema_loc}")



Overriding landing_path to: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/
Overriding checkpoint to: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/
Overriding schema_loc to: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/few_rows/


In [0]:
from pyspark.sql import functions as F

# Databases (Hive metastore style)
spark.sql("CREATE DATABASE IF NOT EXISTS raw")
spark.sql("CREATE DATABASE IF NOT EXISTS governance")

# DQ tables (standardised across raw & curated)
spark.sql("""
CREATE TABLE IF NOT EXISTS governance.dq_run (
  run_id STRING,
  pipeline_name STRING,
  source_name STRING,
  dataset_name STRING,
  layer STRING,
  batch_id STRING,
  triggered_by STRING,
  start_ts TIMESTAMP,
  end_ts TIMESTAMP,
  status STRING,
  stop_on_severity STRING,
  tags MAP<STRING,STRING>
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS governance.dq_check_result (
  run_id STRING,
  check_id STRING,
  check_name STRING,
  layer STRING,
  dataset_name STRING,
  table_name STRING,
  check_ts TIMESTAMP,
  severity STRING,
  action STRING,
  passed BOOLEAN,
  metric_name STRING,
  metric_value DOUBLE,
  threshold_value DOUBLE,
  details STRING
) USING DELTA
""")

print("✅ governance.dq_run and governance.dq_check_result ready")

✅ governance.dq_run and governance.dq_check_result ready


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    BooleanType, DoubleType, MapType
)
from datetime import datetime, timezone

dq_run_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("triggered_by", StringType(), True),
    StructField("start_ts", TimestampType(), True),
    StructField("end_ts", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("stop_on_severity", StringType(), True),
    StructField("tags", MapType(StringType(), StringType()), True),
])

dq_check_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("check_id", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("check_ts", TimestampType(), True),
    StructField("severity", StringType(), True),
    StructField("action", StringType(), True),
    StructField("passed", BooleanType(), True),
    StructField("metric_name", StringType(), True),
    StructField("metric_value", DoubleType(), True),
    StructField("threshold_value", DoubleType(), True),
    StructField("details", StringType(), True),
])


In [0]:
from datetime import datetime, timezone
import uuid

SEVERITY_ORDER = {"INFO": 0, "WARN": 1, "ERROR": 2, "FATAL": 3}

def new_run_id():
    return str(uuid.uuid4())

def severity_ge(a: str, b: str) -> bool:
    return SEVERITY_ORDER.get(a, 0) >= SEVERITY_ORDER.get(b, 0)

def start_run(run_id, layer, batch_id, tags=None):
    tags = tags or {}

    # Make timestamp "naive UTC" so Spark treats it cleanly as TIMESTAMP
    start_ts = datetime.now(timezone.utc).replace(tzinfo=None)

    row = {
        "run_id": run_id,
        "pipeline_name": pipeline_name,
        "source_name": source_name,
        "dataset_name": dataset_name,
        "layer": layer,
        "batch_id": str(batch_id),
        "triggered_by": triggered_by,
        "start_ts": start_ts,
        "end_ts": None,
        "status": "STARTED",
        "stop_on_severity": stop_on_sev,
        "tags": tags
    }

    spark.createDataFrame([row], schema=dq_run_schema) \
         .write.mode("append") \
         .saveAsTable("governance.dq_run")

def end_run(run_id, status):
    spark.sql(f"""
      UPDATE governance.dq_run
      SET end_ts = current_timestamp(), status = '{status}'
      WHERE run_id = '{run_id}'
    """)

def log_check(run_id, check_id, check_name, severity, action, passed,
              metric_name, metric_value, threshold_value, details, layer="raw"):

    check_ts = datetime.now(timezone.utc).replace(tzinfo=None)

    row = {
        "run_id": run_id,
        "check_id": check_id,
        "check_name": check_name,
        "layer": layer,
        "dataset_name": dataset_name,
        "table_name": raw_table,
        "check_ts": check_ts,
        "severity": severity,
        "action": action,
        "passed": bool(passed),
        "metric_name": metric_name,
        "metric_value": float(metric_value) if metric_value is not None else None,
        "threshold_value": float(threshold_value) if threshold_value is not None else None,
        "details": details
    }

    spark.createDataFrame([row], schema=dq_check_schema) \
         .write.mode("append") \
         .saveAsTable("governance.dq_check_result")

def decide_action(passed, severity):
    if passed:
        return "CONTINUE"
    return "STOP" if severity_ge(severity, stop_on_sev) else "WARN"

In [0]:
for col in cfg.get("schema", []):
    t = str(col.get("type", ""))
    if t.lower().startswith("decimal"):
        print("DECIMAL TYPE FOUND:", col["name"], "=>", t)

DECIMAL TYPE FOUND: close_price => decimal(18


In [0]:
from pyspark.sql.types import StringType, DateType, TimestampType, DecimalType

type_map = {
    "string": StringType(),
    "date": DateType(),
    "timestamp": TimestampType(),
}

def parse_type(t: str):
    t = (t or "").strip().lower()

    if t.startswith("decimal"):
        # Accept decimal(18,6) OR decimal(18)
        if "(" in t and ")" in t:
            inside = t[t.find("(")+1:t.find(")")].strip()
            parts = [p.strip() for p in inside.split(",") if p.strip()]
            if len(parts) == 2:
                return DecimalType(int(parts[0]), int(parts[1]))
            elif len(parts) == 1:
                return DecimalType(int(parts[0]), 0)
            else:
                raise ValueError(f"Invalid decimal type: {t}")
        else:
            raise ValueError(f"Invalid decimal type (missing parentheses): {t}")

    return type_map.get(t, StringType())

In [0]:
import re
from pyspark.sql.types import StringType, DateType, TimestampType, DecimalType

type_map = {
    "string": StringType(),
    "date": DateType(),
    "timestamp": TimestampType(),
}

_decimal_re = re.compile(
    r"""^decimal
         (?:\s*\(\s*(\d+)\s*(?:,\s*(\d+)\s*)?\)?   # decimal(18,6) OR decimal(18,6  OR decimal(18)
          |\s+(\d+)\s*(?:,\s*(\d+)\s*)?           # decimal 18,6 OR decimal 18
         )$
     """,
    re.IGNORECASE | re.VERBOSE
)

def parse_type(t: str):
    """
    Robust parsing for schema types coming from YAML.
    Supports:
      - decimal(18,6), decimal(18), decimal(18,6  (missing ')')
      - decimal 18,6, decimal 18
      - string, date, timestamp
    """
    t_raw = (t or "").strip()
    t_norm = t_raw.lower()

    # Decimal handling (tolerant)
    if t_norm.startswith("decimal"):
        m = _decimal_re.match(t_raw)
        if not m:
            raise ValueError(
                f"Invalid decimal type: {t_raw}. Use decimal(p,s) or decimal(p). "
                f"Examples: decimal(18,6), decimal(18)"
            )

        # groups: (p1, s1, p2, s2)
        p = m.group(1) or m.group(3)
        s = m.group(2) or m.group(4) or "0"
        return DecimalType(int(p), int(s))

    # Known simple types
    return type_map.get(t_norm, StringType())


def validate_schema_types(schema_cfg):
    """
    Validate schema types but be tolerant of common decimal formatting issues.
    Returns list of bad (name, type) entries.
    """
    bad = []
    for col in schema_cfg or []:
        t_raw = (col.get("type") or "").strip()
        if t_raw.lower().startswith("decimal"):
            if not _decimal_re.match(t_raw):
                bad.append((col.get("name"), col.get("type")))
    return bad

In [0]:
import re

_decimal_extract = re.compile(r"decimal\s*\(\s*(\d+)\s*(?:,\s*(\d+)\s*)?", re.IGNORECASE)

def normalize_decimal_type(t: str) -> str:
    """
    Normalizes common malformed decimal strings.
    Examples:
      'decimal(18'    -> 'decimal(18,0)'
      'decimal(18,'   -> 'decimal(18,0)'
      'decimal(18,6'  -> 'decimal(18,6)'
      'decimal 18,6'  -> 'decimal(18,6)'
      'decimal(18)'   -> 'decimal(18,0)'
    """
    if t is None:
        return t

    raw = t.strip()
    low = raw.lower().strip()

    if not low.startswith("decimal"):
        return raw

    # Try to extract precision/scale even if missing ')'
    m = _decimal_extract.search(raw)
    if not m:
        return raw  # let validator catch it

    p = m.group(1)
    s = m.group(2) if m.group(2) is not None else "0"
    return f"decimal({int(p)},{int(s)})"


def validate_schema_types(schema_cfg):
    """
    Repairs decimal types in-place and then validates them.
    Returns list of bad (name, type) entries that still can't be parsed.
    """
    bad = []
    for col in schema_cfg or []:
        t = col.get("type")
        if isinstance(t, str) and t.strip().lower().startswith("decimal"):
            fixed = normalize_decimal_type(t)
            col["type"] = fixed  # <-- IMPORTANT: repair the config in place

            # validate final normalized form
            if not re.fullmatch(r"decimal\(\d+,\d+\)", fixed.strip().lower()):
                bad.append((col.get("name"), col.get("type")))
    return bad

In [0]:
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, DateType, TimestampType, DecimalType

# If schema isn't defined due to prior cell order/failure, build it here from cfg
if "schema" not in globals() or schema is None:
    type_map = {
        "string": StringType(),
        "date": DateType(),
        "timestamp": TimestampType(),
    }

    def parse_type(t: str):
        t = (t or "").strip().lower()
        if t.startswith("decimal"):
            # Accept decimal(p,s) or decimal(p)
            if "(" in t and ")" in t:
                inside = t[t.find("(")+1:t.find(")")].strip()
                parts = [p.strip() for p in inside.split(",") if p.strip()]
                if len(parts) == 2:
                    return DecimalType(int(parts[0]), int(parts[1]))
                elif len(parts) == 1:
                    return DecimalType(int(parts[0]), 0)
            # If malformed, default safely rather than crash
            # (or raise if you prefer strict)
            return DecimalType(18, 0)

        return type_map.get(t, StringType())

    schema_fields = []
    for col in cfg.get("schema", []):
        schema_fields.append(
            StructField(
                col["name"],
                parse_type(col.get("type")),
                bool(col.get("nullable", True))
            )
        )

    schema = StructType(schema_fields) if schema_fields else None

print("✅ Schema ready:", schema.simpleString() if schema else "None")

✅ Schema ready: struct<isin:string,trade_date:date,close_price:decimal(18,0),currency:string>


In [0]:
def dq_and_write(microBatchDF, batchId: int):
    run_id = new_run_id()
    start_run(run_id, layer="raw", batch_id=batchId, tags={"stage": "landing_to_raw"})

    try:
        mb = microBatchDF # .persist()

        row_count = mb.count()

        # 1) Missing rows (min expected)
        miss_pol = policies.get("missing_rows", {"severity": "ERROR"})
        passed = row_count >= expected_min_rows
        action = decide_action(passed, miss_pol["severity"])
        log_check(
            run_id=run_id,
            check_id="RAW_ROWCOUNT_MIN",
            check_name="Minimum expected rows present",
            severity=miss_pol["severity"],
            action=action,
            passed=passed,
            metric_name="missing_rows",
            metric_value=max(0, expected_min_rows - row_count),
            threshold_value=expected_min_rows,
            details=f"row_count={row_count}, expected_min_rows={expected_min_rows}"
        )

        # 2) Required columns nulls
        if required_cols:
            agg_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in required_cols]
            nulls_by_col = mb.agg(*agg_exprs).collect()[0].asDict()
            total_required_nulls = sum(nulls_by_col.values())

            req_pol = policies.get("required_nulls", {"severity": "ERROR"})
            passed2 = (total_required_nulls == 0)
            action2 = decide_action(passed2, req_pol["severity"])
            log_check(
                run_id=run_id,
                check_id="RAW_REQUIRED_NULLS",
                check_name="Required fields have no nulls",
                severity=req_pol["severity"],
                action=action2,
                passed=passed2,
                metric_name="null_count_required",
                metric_value=total_required_nulls,
                threshold_value=0,
                details=f"nulls_by_col={nulls_by_col}"
            )

        # 3) Optional null-rate thresholds (typically WARN)
        for col, thr in null_thresholds.items():
            if row_count > 0:
                nulls = mb.select(F.sum(F.when(F.col(col).isNull(), 1).otherwise(0)).alias("n")).collect()[0]["n"]
                null_rate = float(nulls) / float(row_count)
            else:
                nulls, null_rate = row_count, 1.0

            pol = policies.get("null_rate_threshold", {"severity": "WARN"})
            passed3 = null_rate <= float(thr)
            action3 = decide_action(passed3, pol["severity"])
            log_check(
                run_id=run_id,
                check_id=f"RAW_NULLRATE_{col.upper()}",
                check_name=f"Null rate threshold for {col}",
                severity=pol["severity"],
                action=action3,
                passed=passed3,
                metric_name="null_rate",
                metric_value=null_rate,
                threshold_value=float(thr),
                details=f"nulls={nulls}, rows={row_count}"
            )

        # STOP if any STOP action exists
        failing = (spark.table("governance.dq_check_result")
                   .where((F.col("run_id") == run_id) &
                          (F.col("passed") == F.lit(False)) &
                          (F.col("action") == F.lit("STOP")))
                   .count())

        if failing > 0:
            end_run(run_id, "FAILED")
            # raise Exception(f"DQ STOP triggered for run_id={run_id}. See governance.dq_check_result.")
            print(f"⚠️ DQ STOP triggered for run_id={run_id}, batchId={batchId}. Skipping write to raw.")
            return

        # Add run_id column to the batch so it lands in the raw table
        mb_out = mb.withColumn("_run_id", F.lit(run_id))

        # Write to Raw table
        (mb_out.write
          .format("delta")
          .mode("append")
          .option("mergeSchema", "true")
          .saveAsTable(raw_table))

        end_run(run_id, "SUCCEEDED")

    except Exception as e:
        try:
            end_run(run_id, "FAILED")
        except:
            pass
        raise
    finally:
        try:
            mb.unpersist()
        except:
            pass

In [0]:
print("Has cfg?      ", "cfg" in globals())
print("Has schema?   ", "schema" in globals())
print("Has reader?   ", "reader" in globals())
print("Has df?       ", "df" in globals())

print("landing_path: ", landing_path if "landing_path" in globals() else None)
print("checkpoint:   ", checkpoint if "checkpoint" in globals() else None)
print("schema_loc:   ", schema_loc if "schema_loc" in globals() else None)
print("fmt:          ", fmt if "fmt" in globals() else None)
print("Files in landing_path:")
display(dbutils.fs.ls(landing_path))


Has cfg?       True
Has schema?    True
Has reader?    True
Has df?        True
landing_path:  abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/
checkpoint:    abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/
schema_loc:    abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/few_rows/
fmt:           csv
Files in landing_path:


path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/few_rows_20260130_1558.csv,few_rows_20260130_1558.csv,2133,1769788707000


In [0]:
display(landing_path)
#print("✅ Schema ready:", schema.simpleString() if schema else "None")

'abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/'

In [0]:
from pyspark.sql import functions as F

# ---- 0) Ensure required config variables exist ----
needed = ["landing_path", "schema_loc", "fmt", "read_opts", "checkpoint", "run_mode", "processing_time"]
missing = [v for v in needed if v not in globals()]
if missing:
    raise NameError(
        f"Missing variables: {missing}. Run the config/YAML load cells first."
    )

# ---- 1) Ensure reader exists ----
if "reader" not in globals():
    reader = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", fmt)
        .option("cloudFiles.schemaLocation", schema_loc)
    )
    # Apply read options from config
    for k, v in read_opts.items():
        reader = reader.option(k, v)

# ---- 2) Apply explicit schema (if present) ----
if "schema" in globals() and schema is not None:
    reader = reader.schema(schema)

# ---- 3) Build df ----
df = (reader.load(landing_path)
      .withColumn("_ingest_ts", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path")))

print("✅ df created")

# ---- 4) Start writer ----
writer = (df.writeStream
  .foreachBatch(dq_and_write) # Ensure dq_and_write is defined before this line
  .option("checkpointLocation", checkpoint)
)

if run_mode.lower() == "processingtime":
    writer = writer.trigger(processingTime=processing_time)
else:
    writer = writer.trigger(availableNow=True)

q = writer.start()
q.awaitTermination()

print("✅ Ingestion completed")

✅ df created
⚠️ DQ STOP triggered for run_id=23538e11-02b6-4f98-8ec2-feededd40aa1, batchId=0. Skipping write to raw.
✅ Ingestion completed


In [0]:
display(dbutils.fs.ls(f"{checkpoint.rstrip('/')}/commits/"))

path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/commits/0,0,29,1769789369000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/commits/__tmp_path_dir/,__tmp_path_dir/,0,1769789369000


In [0]:
print(dbutils.fs.head(f"{checkpoint.rstrip('/')}/commits/0", 5000))


v1
{"nextBatchWatermarkMs":0}


In [0]:
display(dbutils.fs.ls(f"{checkpoint.rstrip('/')}/offsets/"))

path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/offsets/0,0,1639,1769789361000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/offsets/__tmp_path_dir/,__tmp_path_dir/,0,1769789360000


In [0]:
print(dbutils.fs.head(f"{checkpoint.rstrip('/')}/offsets/0", 20000))

v1
{"batchWatermarkMs":0,"batchTimestampMs":1769789360891,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.databricks.optimizer.joinOrToUnionAllExpansionWithEquiJoin.streaming":"false","spark.sql.streaming.dropDuplicates.stateFormatVersion":"1","spark.databricks.streaming.disableChangingStatelessShufflePartitionsForExistingQuery":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.databricks.optimizer.convertCollectSetToCollectlistDistinct.enableStreami

In [0]:
print("checkpoint:", checkpoint)
display(dbutils.fs.ls(checkpoint))

checkpoint: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/


path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/__tmp_path_dir/,__tmp_path_dir/,0,1769789358000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/commits/,commits/,0,1769789359000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/metadata,metadata,45,1769789358000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/offsets/,offsets/,0,1769789359000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/few_rows/sources/,sources/,0,1769789358000
